In [ ]:
# === TRAINING PRESETS ===
# Dataset: 220,000 samples available

PRESET = 2

configs = {
    0: {  # Ultra-Fast Debug - test code changes
        'TRAIN_DATA_SIZE': 2048 * 32,  # ~20 updates
        'N_ENVS': 8,          # Optimal for most CPUs (8-16 cores)
        'N_STEPS': 2048,      # Standard PPO rollout length
        'BATCH_SIZE': 256,    # Larger batches = more stable gradients
        'NUM_ITERATIONS': 4,
    },
    1: {  # Full Training - production
        'TRAIN_DATA_SIZE': 2048 * 64,
        'N_ENVS': 8,
        'N_STEPS': 2048,
        'BATCH_SIZE': 256,    # Larger batches = more stable gradients
        'NUM_ITERATIONS': 4,
    },
    2: {  # Extended Training
        'TRAIN_DATA_SIZE': 131_072,
        'N_ENVS': 8,
        'N_STEPS': 2048,
        'BATCH_SIZE': 256,    # Larger batches = more stable gradients
        'NUM_ITERATIONS': 6,
    },
    3: {  # Max Training
        'TRAIN_DATA_SIZE': 131_072 + 65_536,
        'N_ENVS': 8,
        'N_STEPS': 2048,
        'BATCH_SIZE': 256,    # Larger batches = more stable gradients
        'NUM_ITERATIONS': 8,
    },
}

# Load config
cfg = configs[PRESET]
TRAIN_DATA_SIZE = cfg['TRAIN_DATA_SIZE']
N_ENVS = cfg['N_ENVS']
N_STEPS = cfg['N_STEPS']
NUM_ITERATIONS = cfg['NUM_ITERATIONS']
BATCH_SIZE = cfg['BATCH_SIZE']

# Fixed params
LOOKBACK_WINDOW = 288
HIDDEN_DIM = 128
POLICY_LAYERS = [256, 128]
#POLICY_LAYERS = [512, 256, 128]
#VALUE_LAYERS = [128, 64]
VALUE_LAYERS = [256, 128]

# Hyperparameters - OPTIMIZED FOR STABLE LEARNING
LEARNING_RATE_START = 2e-4      # Standard PPO
LEARNING_RATE_DECAY = 0.0       # Constant LR (or remove lambda entirely)
N_EPOCHS = 3                    # Reduce to prevent overfitting
ENT_COEF = 0.2                  # HIGHER entropy to force exploration
CLIP_RANGE = 0.15                 # Good
CLIP_RANGE_VF = 0.2             # Good
TARGET_KL = 0.05                # More conservative KL divergence
VF_COEF = 0.5                   # Reduce value function weight
MAX_GRAD_NORM = 0.5              # Good
USE_SDE = False                  # Enable stochastic actions
GAE_LAMBDA = 0.95                # Good
GAMMA = 0.99                     # Good

DATA_SYMBOL = 'BTCUSDT'
DATA_TIMEFRAME = '5m'
DATA_PATH = f'data/binance-{DATA_SYMBOL}-{DATA_TIMEFRAME}.pkl'
MODEL_SAVE_PATH = "trading_bot"

# Calculated
TOTAL_TIMESTEPS = TRAIN_DATA_SIZE * NUM_ITERATIONS

print(f"Preset {PRESET}: {TOTAL_TIMESTEPS:,} steps | {NUM_ITERATIONS} iters | {N_ENVS} envs | {TRAIN_DATA_SIZE:,} samples")
print(f"\n🎯 TRAINING CONFIGURATION:")
print(f"   • {N_ENVS} parallel envs (optimal for CPU)")
print(f"   • {N_STEPS} steps per rollout ({N_ENVS * N_STEPS:,} samples per update)")
print(f"   • Batch size: {BATCH_SIZE} ({N_ENVS * N_STEPS // BATCH_SIZE} minibatches)")
print(f"   • VP cache: Shared across envs (87.5% speedup)")
print(f"   • Entropy coef: {ENT_COEF} (balanced exploration)")
print(f"   • TARGET_KL: {TARGET_KL} (allows larger policy updates)")
print(f"   • Learning rate: {LEARNING_RATE_START}")

Preset 2: 786,432 steps | 6 iters | 8 envs | 131,072 samples

🎯 TRAINING CONFIGURATION:
   • 8 parallel envs (optimal for CPU)
   • 2048 steps per rollout (16,384 samples per update)
   • Batch size: 256 (64 minibatches)
   • VP cache: Shared across envs (87.5% speedup)
   • Entropy coef: 0.2 (balanced exploration)
   • TARGET_KL: 0.05 (allows larger policy updates)
   • Learning rate: 0.0002


In [2]:
import warnings
warnings.filterwarnings('ignore', message='enable_nested_tensor is True')
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.monitor import Monitor
from stable_baselines3 import PPO
import torch
import torch.nn as nn
import pandas as pd
import time
import psutil
import os
from environments.simple_trading_env import SimpleTradingEnv
from environments.trading_hybrid_extractor import TradingHybridExtractor

# Load data
df = pd.read_pickle(DATA_PATH)
train_data = df.iloc[0:TRAIN_DATA_SIZE].reset_index(drop=True)
print(f"Loaded {len(df):,} rows | Training on {len(train_data):,} samples")

# 🚀 Create shared VP cache that all 8 environments will use
# This prevents redundant VP calculations across parallel envs
shared_vp_cache = {}
print(f"Created shared VP cache (will be populated during training)")

# Setup policy - HYBRID MULTI-SCALE CNN Architecture
# Multi-scale CNNs: Capture candlestick patterns at different timeframes
# Volume Profile: Market structure via volume distribution (POC/VAH/VAL)
# Clean & Fast: Pure tensor operations, no CPU bottlenecks
policy_kwargs = dict(
    features_extractor_class=TradingHybridExtractor,
    features_extractor_kwargs=dict(hidden_dim=HIDDEN_DIM),
    net_arch=dict(pi=POLICY_LAYERS, vf=VALUE_LAYERS),
    activation_fn=torch.nn.ReLU,
    ortho_init=False,
)

# Create environments - all sharing the same VP cache
vec_env = make_vec_env(
    lambda: Monitor(SimpleTradingEnv(
        train_data, 
        lookback_window=LOOKBACK_WINDOW,
        taker_commission=0.0, # No commissions for training
        maker_commission=0.0, # No commissions for training
        vp_cache=shared_vp_cache,
        enable_trade_logging=False
    )),
    n_envs=N_ENVS
)

print(f"\n📊 Environment info:")
print(f"   Number of envs: {vec_env.num_envs}")
process = psutil.Process(os.getpid())
ram_gb = process.memory_info().rss / (1024 ** 3)
print(f"   RAM usage: {ram_gb:.2f} GB\n")



# Create model
model = PPO(
    "MultiInputPolicy",
    vec_env,
    device="cuda",
    learning_rate=lambda f: LEARNING_RATE_START * (1 - LEARNING_RATE_DECAY * f),
    n_steps=N_STEPS,
    batch_size=BATCH_SIZE,
    n_epochs=N_EPOCHS,
    gamma=GAMMA,
    gae_lambda=GAE_LAMBDA,
    clip_range=CLIP_RANGE,
    ent_coef=ENT_COEF,
    vf_coef=VF_COEF,
    max_grad_norm=MAX_GRAD_NORM,
    target_kl=TARGET_KL,
    stats_window_size=LOOKBACK_WINDOW,
    policy_kwargs=policy_kwargs,
    clip_range_vf=CLIP_RANGE_VF,
    use_sde=USE_SDE,
    tensorboard_log="./tensorboard_logs/",
    verbose=2
)

import os
if os.path.exists("trading_bot.zip"):
    os.remove("trading_bot.zip")

# Load pre-trained model
#model = PPO.load(f"{MODEL_SAVE_PATH}_baseline", env=vec_env, device="cuda")
#model.target_kl = TARGET_KL
#model.n_epochs = N_EPOCHS
#print(f"\n✅ Loaded pre-trained model: {MODEL_SAVE_PATH}_baseline")


try:
    # Train
    print(f"Starting training: {TOTAL_TIMESTEPS:,} steps...")
    torch.cuda.empty_cache()
    model.learn(total_timesteps=TOTAL_TIMESTEPS, progress_bar=True)
except KeyboardInterrupt:
    print("\n⏸ Training interrupted by user")
    
# Save
model.save(f"{MODEL_SAVE_PATH}")
print(f"\n✓ Saved: {MODEL_SAVE_PATH}")

print(f"\n✅ Training complete!")
print(f"📊 VP Cache stats: {len(shared_vp_cache):,} unique steps cached")
print(f"💾 Final RAM usage: {psutil.Process(os.getpid()).memory_info().rss / (1024 ** 3):.2f} GB")

# Check action distribution
print("\n📊 Action Distribution Check:")
obs = vec_env.reset()
action_counts = {0: 0, 1: 0, 2: 0}
for _ in range(1000):
    action, _ = model.predict(obs, deterministic=False)
    for a in action:
        action_counts[int(a.item())] += 1

total = sum(action_counts.values())
print(f"   HOLD:  {action_counts[0]/total*100:.1f}%")
print(f"   LONG:  {action_counts[1]/total*100:.1f}%")
print(f"   SHORT: {action_counts[2]/total*100:.1f}%")

Loaded 264,323 rows | Training on 131,072 samples
Created shared VP cache (will be populated during training)
Info: Dropped 99 rows due to NaNs after adding indicators.
Info: Dropped 99 rows due to NaNs after adding indicators.
Info: Dropped 99 rows due to NaNs after adding indicators.
Info: Dropped 99 rows due to NaNs after adding indicators.
Info: Dropped 99 rows due to NaNs after adding indicators.
Info: Dropped 99 rows due to NaNs after adding indicators.
Info: Dropped 99 rows due to NaNs after adding indicators.
Info: Dropped 99 rows due to NaNs after adding indicators.

📊 Environment info:
   Number of envs: 8
   RAM usage: 0.86 GB

Using cuda device


ValueError: generalized State-Dependent Exploration (gSDE) can only be used with continuous actions.

# 🎨 Real-Time Observation Monitor

Launch the Streamlit dashboard to monitor observations in real-time without any disk writes!

### Alternative: Launch from Terminal

If you prefer to launch manually, run this in terminal:

```bash
streamlit run streamlit_obs_monitor.py
```

Then open: http://localhost:8501